In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers torch scikit-learn networkx
!pip install -q sinling sentencepiece accelerate
!python -m pip install "pip<24.1"
!pip install omegaconf==2.0.6 hydra-core==1.0.7
!pip install git+https://github.com/One-sixth/fairseq.git
!pip install --upgrade laser_encoders

print("Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 37.8 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.8/123.8 kB 5.2 MB/s eta 0:00:00
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.8-py3-none-any.whl size=141211 sha256=9145f79db124acaeb14b9c9ba57237663dc5cc748e006ca563e73e6365d0995c
  Stored in directory: /root/.cache/pip/wheels/3e/92/b7/08c6a108fc5bf6370a7540d11bbe9befc99b7e045ac7558d49
Successfully built antlr4-python3-runtime
DEPRECATION:

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 895.1/895.1 kB 25.1 MB/s eta 0:00:00
  Created wheel for unicategories: filename=unicategories-0.1.2-py2.py3-none-any.whl size=31656 sha256=24518da318e313b7415c2ed0519dcbd21c7875cd8a282816f4a70f4462247940
  Stored in directory: /root/.cache/pip/wheels/d0/19/7f/6d82168bd56dd6bffa309daf231a2e19ec74bbc68f78666d1d
Successfully built unicategories
Dependencies installed


In [3]:
import os
import json
import torch
import torch.nn as nn
import numpy as np
import networkx as nx
from sinling import SinhalaTokenizer, POSTagger
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
from laser_encoders import LaserEncoderPipeline
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported")

Libraries imported


In [ ]:
BASE_PATH      = "YOUR_PATH"
TEST_PATH      = f"{BASE_PATH}/YOUR_PATH"
MODEL_PATH     = f"{BASE_PATH}/YOUR_PATH"
OUTPUT_PATH    = f"{BASE_PATH}/OutYOUR_PATHputs"
SUMMARY_OUTPUT = f"{OUTPUT_PATH}/YOUR_PATH"

os.makedirs(SUMMARY_OUTPUT, exist_ok=True)

assert os.path.exists(TEST_PATH),  f"Test path not found: {TEST_PATH}"
assert os.path.exists(MODEL_PATH), f"Model path not found: {MODEL_PATH}"

print("Paths configured:")
print(f"   Test docs: {TEST_PATH}")
print(f"   Model:     {MODEL_PATH}")
print(f"   Output:    {SUMMARY_OUTPUT}")

Paths configured:
   Test docs: /content/drive/MyDrive/SinBrief_Implementation/Dataset/Test
   Model:     /content/drive/MyDrive/SinBrief_Implementation/Models
   Output:    /content/drive/MyDrive/SinBrief_Implementation/Outputs/Laser_Summaries


In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

tokenizer_sin = SinhalaTokenizer()
pos_tagger    = POSTagger()

LEGAL_KEYWORDS = [
    'අධිකරණය',
    'නීතිය',
    'විනිශ්චය',
    'දඬුවම',
    'චෝදනා',
    'සාක්ෂි',
    'තීන්දුව',
    'අභියාචනය',
    'නඩුව',
    'වරද',
    'දෝෂ',
    'බලය',
    'කොන්දේසි',
    'ප්‍රතිපාදන',
    'ව්‍යවස්ථා',
    'පනත',
    'පැමිණිලිකරු',
    'කොන්ත්‍රාත්තුව',
    'පැමිණිල්ල'
]

print(f"Tools initialized")
print(f"Loaded {len(LEGAL_KEYWORDS)} legal keywords")

Using device: cuda
Tools initialized
Loaded 19 legal keywords


In [6]:
import argparse
torch.serialization.add_safe_globals([argparse.Namespace])

print("Loading LASER encoder for Sinhala...")
laser_encoder = LaserEncoderPipeline(lang="sin_Sinh")
print("LASER encoder loaded")

Loading LASER encoder for Sinhala...


100%|██████████| 1.01M/1.01M [00:00<00:00, 24.8MB/s]
100%|██████████| 608M/608M [00:13<00:00, 44.6MB/s]
100%|██████████| 470k/470k [00:00<00:00, 14.0MB/s]


LASER encoder loaded


In [7]:
def cluster_sentences(sentences, num_clusters=3):
    """Cluster sentences by semantic similarity using LASER embeddings"""
    if len(sentences) <= num_clusters:
        return [[sent] for sent in sentences]

    print(f"   Generating embeddings for {len(sentences)} sentences...")
    embeddings = laser_encoder.encode_sentences(sentences)   # (N, 1024)

    clustering = AgglomerativeClustering(
        n_clusters=min(num_clusters, len(sentences)),
        metric='cosine',
        linkage='average'
    )
    labels = clustering.fit_predict(embeddings)

    clusters = {}
    for idx, label in enumerate(labels):
        if label not in clusters:
            clusters[label] = []
        clusters[label].append(sentences[idx])

    return list(clusters.values())

print("Clustering function defined")

Clustering function defined


In [8]:
KEEP_POS = {'NOUN', 'VERB', 'AUX', 'PROPN', 'ADJ'}

def build_word_graph_weighted(sentences_in_cluster, legal_keywords):
    G = nx.DiGraph()
    G.add_node("START")
    G.add_node("END")

    for sentence in sentences_in_cluster:
        tokenized = tokenizer_sin.tokenize(sentence)
        if len(tokenized) == 0:
            continue

        words_with_pos = []
        try:
            pos_tags = pos_tagger.predict([tokenized])[0]
            words_with_pos = [(w, p) for w, p in zip(tokenized, pos_tags)
                             if p in KEEP_POS and len(w) > 1]
        except:
            pass

        if len(words_with_pos) < 3:
            words_with_pos = [(w, 'NOUN') for w in tokenized if len(w) > 2]

        words_with_pos = words_with_pos[:30]

        if len(words_with_pos) == 0:
            continue

        first_word = words_with_pos[0]
        weight = 2.0 if any(kw in first_word[0] for kw in legal_keywords) else 1.0
        if G.has_edge("START", first_word):
            G["START"][first_word]['weight'] += weight
        else:
            G.add_edge("START", first_word, weight=weight)

        for i in range(len(words_with_pos)-1):
            current   = words_with_pos[i]
            next_word = words_with_pos[i+1]
            has_keyword = any(kw in current[0] or kw in next_word[0] for kw in legal_keywords)
            weight = 2.0 if has_keyword else 1.0
            if G.has_edge(current, next_word):
                G[current][next_word]['weight'] += weight
            else:
                G.add_edge(current, next_word, weight=weight)

        last_word = words_with_pos[-1]
        weight = 2.0 if any(kw in last_word[0] for kw in legal_keywords) else 1.0
        if G.has_edge(last_word, "END"):
            G[last_word]["END"]['weight'] += weight
        else:
            G.add_edge(last_word, "END", weight=weight)

    print(f"       Built graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    return G

print("Graph building function defined")

Graph building function defined


In [9]:
import random

def generate_fused_sentences(G, num_paths=5, max_length=25, min_length=8):
    if G.number_of_nodes() <= 2:
        return []

    candidates     = []
    seen_word_sets = set()

    for attempt in range(num_paths * 5):
        try:
            current = "START"
            words   = []

            for step in range(max_length + 5):
                neighbors = list(G.successors(current))

                if len(words) < min_length and "END" in neighbors:
                    neighbors.remove("END")

                if len(neighbors) == 0:
                    break

                if len(words) >= min_length and "END" in neighbors:
                    if random.random() < 0.3:
                        break

                valid_neighbors = [n for n in neighbors if n != "END"]
                weights         = [G[current][n].get('weight', 1.0) for n in valid_neighbors]

                if len(valid_neighbors) == 0:
                    break

                next_node = random.choices(valid_neighbors, weights=weights)[0]

                if isinstance(next_node, tuple):
                    word = next_node[0]
                    if len(word) > 1:
                        words.append(word)

                current = next_node

                if len(words) >= max_length:
                    break

            if len(words) < min_length or len(words) > max_length:
                continue

            word_set = frozenset(words)
            if word_set in seen_word_sets:
                continue

            seen_word_sets.add(word_set)
            fused = ' '.join(words)

            if len(fused) < 20:
                continue

            if len(set(words)) < len(words) * 0.6:
                continue

            candidates.append(fused)

            if len(candidates) >= num_paths:
                break

        except:
            continue

    if len(candidates) < num_paths:
        word_weights = {}
        for u, v, data in G.edges(data=True):
            for node in [u, v]:
                if isinstance(node, tuple) and node not in ["START", "END"]:
                    word = node[0]
                    if len(word) > 1:
                        word_weights[word] = word_weights.get(word, 0) + data.get('weight', 1.0)

        if len(word_weights) >= min_length:
            sorted_words = sorted(word_weights.items(), key=lambda x: x[1], reverse=True)
            attempts = 0
            while len(candidates) < num_paths and attempts < 10:
                attempts += 1
                n_words = random.randint(min_length, min(max_length, len(sorted_words)))
                if attempts == 1:
                    selected = [w for w, _ in sorted_words[:n_words]]
                else:
                    top_k     = len(sorted_words) // 2
                    top_words = [w for w, _ in sorted_words[:top_k]]
                    selected  = random.sample(top_words, min(n_words, len(top_words)))

                word_set = frozenset(selected)
                if word_set in seen_word_sets:
                    continue
                seen_word_sets.add(word_set)
                fused = ' '.join(selected)
                if len(fused) >= 20:
                    candidates.append(fused)

    return candidates[:num_paths]


def filter_low_quality_candidates(candidates):
    from collections import Counter
    filtered = []
    for candidate in candidates:
        if len(candidate) < 20:
            continue
        words = candidate.split()
        if len(set(words)) / len(words) < 0.6:
            continue
        if max(Counter(words).values()) >= 3:
            continue
        filtered.append(candidate)
    return filtered if len(filtered) > 0 else candidates

print("Path generation and filtering functions defined")

Path generation and filtering functions defined


In [10]:
class LegalSentenceScorer_LASER(nn.Module):
    def __init__(self, embedding_dim=1024, dropout=0.3):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, embedding):
        return self.classifier(embedding).squeeze()

scorer_model = LegalSentenceScorer_LASER()
checkpoint   = torch.load(f"{MODEL_PATH}/laser_sentence_scorer.pt", map_location=device)
scorer_model.load_state_dict(checkpoint['classifier_state_dict'])
scorer_model.to(device)
scorer_model.eval()

print(f"LASER scorer loaded (Val Accuracy: {checkpoint['best_val_accuracy']:.4f})")

LASER scorer loaded (Val Accuracy: 0.7737)


In [11]:
def generate_summary_with_scoring(document, num_clusters=3):
    import re
    split_pattern = re.compile(r'[.!?;]+')
    sentences = [s.strip() for s in split_pattern.split(document) if len(s.strip()) > 20]

    if len(sentences) < 3:
        return '. '.join(sentences) + '.'

    print(f"   Clustering {len(sentences)} sentences...")
    clusters = cluster_sentences(sentences, num_clusters=min(3, max(2, len(sentences)//5)))
    print(f"   Created {len(clusters)} clusters")

    summary_sentences = []

    for cluster_idx, cluster in enumerate(clusters, 1):
        if len(cluster) == 0:
            continue

        print(f"   Processing cluster {cluster_idx}/{len(clusters)} ({len(cluster)} sentences)...")

        G = build_word_graph_weighted(cluster, LEGAL_KEYWORDS)

        if G.number_of_nodes() <= 10:
            print(f"       Graph too small, skipping")
            continue

        candidates = generate_fused_sentences(G, num_paths=8, max_length=25, min_length=8)

        if len(candidates) == 0:
            print(f"       No candidates generated, skipping")
            continue

        candidates = filter_low_quality_candidates(candidates)

        if len(candidates) == 0:
            print(f"       All candidates filtered, skipping")
            continue

        print(f"     Scoring {len(candidates)} candidates with LASER...")

        embeddings = laser_encoder.encode_sentences(candidates)
        emb_tensor = torch.tensor(embeddings, dtype=torch.float32).to(device)

        with torch.no_grad():
            scores = scorer_model(emb_tensor)

        if scores.dim() == 0:
            scores = scores.unsqueeze(0)

        scores_list      = scores.cpu().numpy().tolist()
        candidate_scores = list(zip(candidates, scores_list))

        for cand_idx, (candidate, score) in enumerate(candidate_scores):
            print(f"       Candidate {cand_idx+1}: {score:.4f} | {candidate[:60]}...")

        candidate_scores.sort(key=lambda x: x[1], reverse=True)

        for i in range(min(2, len(candidate_scores))):
            candidate, score = candidate_scores[i]
            print(f"     Selected #{i+1} (score: {score:.4f})")
            summary_sentences.append(candidate)

    if len(summary_sentences) == 0:
        return '. '.join(sentences[:3]) + '.'

    summary    = '. '.join(summary_sentences)
    target_min = 300
    target_max = 500

    if len(summary) <= target_max:
        if not summary.endswith('.'):
            summary += '.'
    else:
        parts         = [s.strip() for s in summary.split('.') if s.strip()]
        final_summary = ""
        for sent in parts:
            test = final_summary + sent + '. '
            if len(test) > target_max:
                if len(final_summary) > target_min:
                    break
                else:
                    final_summary = test
                    break
            else:
                final_summary = test
        summary = final_summary.strip()
        if not summary.endswith('.'):
            summary += '.'

    if not summary.endswith('.'):
        summary += '.'

    return summary

print("Summary generation function defined")

Summary generation function defined


In [12]:
print("Loading test documents...")
test_files = [f for f in os.listdir(TEST_PATH) if f.endswith('.txt')]
print(f"Found {len(test_files)} test documents")

print("\n" + "="*60)
print("GENERATING SUMMARIES")
print("="*60 + "\n")

successful   = 0
failed       = 0
failed_files = []

for i, filename in enumerate(test_files, 1):
    print(f"\n[{i}/{len(test_files)}] Processing: {filename}")
    print("-" * 60)

    try:
        filepath = os.path.join(TEST_PATH, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            document = f.read().strip()

        if len(document) < 100:
            print("     Document too short, skipping")
            failed += 1
            failed_files.append(filename)
            continue

        summary     = generate_summary_with_scoring(document, num_clusters=3)
        base_name   = os.path.splitext(filename)[0]
        output_file = os.path.join(SUMMARY_OUTPUT, f"{base_name}_summary.txt")

        with open(output_file, 'w', encoding='utf-8') as out:
            out.write(summary)

        print(f"\n    Summary generated ({len(summary)} chars)")
        print(f"    Preview: {summary[:150]}...")
        successful += 1

    except Exception as e:
        print(f"    Error: {e}")
        failed += 1
        failed_files.append(filename)

print("\n" + "="*60)
print("SUMMARY GENERATION COMPLETE")
print("="*60)
print(f"\n Successfully generated: {successful} summaries")
print(f" Failed: {failed} documents")
if failed_files:
    print(f"\n Failed files:")
    for f in failed_files[:10]:
        print(f"   - {f}")
print(f"\n Summaries saved to: {SUMMARY_OUTPUT}")

Streaming output truncated to the last 5000 lines.
       Candidate 2: 0.0872 | III විෂය රක්ෂණයට අනෙකුත් රක්ෂණ රක්ෂිත නිෂ්පාදන පැලෑටි ඖෂධීය...
     Selected #1 (score: 0.0924)
     Selected #2 (score: 0.0872)
   Processing cluster 3/3 (1 sentences)...
       Built graph: 8 nodes, 8 edges
       Graph too small, skipping

    Summary generated (488 chars)
    Preview: භාරගනු ලැබුවා සියලු අංශ සම්බන්ධයෙන් අනුගමනය යුතු අතර ගැසට් පත්‍රයේ කරනු ලබන නියමයක් මගින් පිහිටුවන කෘෂිකර්ම විෂයය භාර අමාත්‍යවරයාගේ අමාත්‍යාංශයේ ලේකම්...

[102/242] Processing: acts_2003-07-30_Mediation_Special_Categories_of_Disputes_si.txt
------------------------------------------------------------
   Clustering 91 sentences...
   Generating embeddings for 91 sentences...
   Created 3 clusters
   Processing cluster 1/3 (88 sentences)...
       Built graph: 637 nodes, 1494 edges
     Scoring 7 candidates with LASER...
       Candidate 1: 0.1339 | මූල මණ්ඩලයේ සභාපතිවරයා විසින් සිවිල් අධිකරණයක් විසින් කරන...
       Candidat

In [13]:
stats = {
    'total_test_documents':  len(test_files),
    'successful_summaries':  successful,
    'failed_documents':      failed,
    'success_rate':          successful / len(test_files) * 100 if len(test_files) > 0 else 0,
    'output_directory':      SUMMARY_OUTPUT,
    'scorer_model':          'LASER (frozen encoder + classifier)',
    'modifications': [
        'Legal keyword-weighted graph construction',
        'Multi-path candidate generation with LASER neural scoring'
    ],
    'parameters': {
        'num_clusters':          3,
        'num_paths_per_cluster': 5,
        'max_sentence_length':   20,
        'summary_max_length':    300,
        'laser_language':        'sin_Sinh',
        'laser_embedding_dim':   1024
    }
}

stats_file = f"{OUTPUT_PATH}/summary_generation_stats_laser.json"
with open(stats_file, 'w', encoding='utf-8') as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print(f"Statistics saved to {stats_file}")

Statistics saved to /content/drive/MyDrive/SinBrief_Implementation/Outputs/summary_generation_stats_laser.json
